# Week 1: Tokenization Analysis

This notebook measures the multilingual tokenization difference between a Spanish passage and a faithful English translation using the two encodings required by the Week 1 assignment: `cl100k_base` and `o200k_base`.

The analysis focuses on token counts, context-window impact, cost implications, individual word splits, and one counterintuitive tokenization case.


In [1]:
# Setup. In Colab, uncomment the install line on first run.
# !pip install tiktoken

import os
import pathlib
import tiktoken

os.environ["TIKTOKEN_CACHE_DIR"] = str(
    (pathlib.Path(".") / ".tiktoken_cache").resolve()
)
os.makedirs(os.environ["TIKTOKEN_CACHE_DIR"], exist_ok=True)

gpt4 = tiktoken.get_encoding("cl100k_base")   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding("o200k_base")  # GPT-4o

print(
    "tiktoken",
    tiktoken.__version__,
    "- encoders ready (cl100k_base, o200k_base)",
)


tiktoken 0.14.0 - encoders ready (cl100k_base, o200k_base)


## Part 1: Repository

The course repository is public and organized by week. It includes a project `README.md` and AI-tool context files, including `GEMINI.md` for Gemini. This work is being completed on the `week-01/tokenization-assignment` branch and submitted through a pull request into `main`.


## Helpers

The assignment provides two small helper functions: one counts tokens for a string, and the other displays the exact sub-token pieces produced by an encoder.


In [2]:
def count_tokens(text, enc):
    return len(enc.encode(text))


def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([token_id]) for token_id in ids]
    print(f"{word!r:24s} -> {len(ids)} token(s): {pieces}")


# Quick demonstration of how differently shaped strings can tokenize.
for word in ["Panic", " towel", "42", "antidisestablishmentarianism"]:
    show_split(word)


'Panic'                  -> 2 token(s): ['P', 'anic']
' towel'                 -> 1 token(s): [' towel']
'42'                     -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Spanish and English passages

The Spanish passage below was prepared specifically for this analysis. The English passage is a faithful translation of the same meaning rather than a separately selected passage. The Spanish passage exceeds the assignment's 100-word minimum.


In [3]:
# Generated by ChatGPT

english_text = """Language models can seem simple when a person writes a question and receives an answer, but the internal process depends on many technical decisions. Before the model interprets a sentence, the text is divided into tokens. Those tokens represent words, parts of words, punctuation marks, and other fragments. The way a tokenizer divides text affects how much space a conversation occupies inside the context window and how much a request can cost. This effect is not the same for every language. A message in Spanish may require a different number of tokens than an equivalent message in English. For that reason, measuring tokenization directly helps us understand how an apparently small design decision can influence cost, efficiency, and the experience of users who speak different languages."""

spanish_text = """Los modelos de lenguaje pueden parecer simples cuando una persona escribe una pregunta y recibe una respuesta, pero el proceso interno depende de muchas decisiones técnicas. Antes de que el modelo interprete una oración, el texto se divide en tokens. Esos tokens representan palabras, partes de palabras, signos de puntuación y otros fragmentos. La forma en que un tokenizador divide el texto afecta cuánto espacio ocupa una conversación dentro de la ventana de contexto y cuánto puede costar una solicitud. Este efecto no es igual para todos los idiomas. Un mensaje en español puede requerir una cantidad diferente de tokens que un mensaje equivalente en inglés. Por eso, medir la tokenización directamente ayuda a comprender cómo una decisión de diseño aparentemente pequeña puede influir en el costo, la eficiencia y la experiencia de usuarios que hablan distintos idiomas."""

english_words = len(english_text.split())
spanish_words = len(spanish_text.split())

print("English words:", english_words)
print("Spanish words:", spanish_words)

assert spanish_words >= 100, "The Spanish passage must contain at least 100 words."


def report(label, text):
    print(
        f"{label:9s} | chars {len(text):4d} "
        f"| GPT-4 {count_tokens(text, gpt4):4d} "
        f"| GPT-4o {count_tokens(text, gpt4o):4d}"
    )


report("English", english_text)
report("Spanish", spanish_text)

tax_gpt4 = count_tokens(spanish_text, gpt4) / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(spanish_text, gpt4o) / count_tokens(english_text, gpt4o)

print(
    f"\nMultilingual tax | GPT-4: {tax_gpt4:.2f}x "
    f"| GPT-4o: {tax_gpt4o:.2f}x"
)

print(
    f"\nInterpretation: For this passage, Spanish uses {tax_gpt4:.2f} times "
    "as many cl100k_base tokens as the English translation and "
    f"{tax_gpt4o:.2f} times as many o200k_base tokens. "
    "The comparison shows that the multilingual tax depends on the tokenizer, "
    "not only on the meaning of the text."
)


English words: 126
Spanish words: 138
English   | chars  796 | GPT-4  143 | GPT-4o  143
Spanish   | chars  876 | GPT-4  197 | GPT-4o  160

Multilingual tax | GPT-4: 1.38x | GPT-4o: 1.12x

Interpretation: For this passage, Spanish uses 1.38 times as many cl100k_base tokens as the English translation and 1.12 times as many o200k_base tokens. The comparison shows that the multilingual tax depends on the tokenizer, not only on the meaning of the text.


## Part 3: Cost and context-window impact

Token counts affect both how much text fits into a context window and how much token-metered inference costs. The following cell converts the measured counts into those engineering consequences for a 128,000-token context window.


In [4]:
CTX = 128_000

for encoder_name, encoder in [
    ("cl100k_base", gpt4),
    ("o200k_base", gpt4o),
]:
    en = count_tokens(english_text, encoder)
    es = count_tokens(spanish_text, encoder)
    multiplier = es / en

    print(f"\n{encoder_name}")
    print(f"  English passage: {en} tokens")
    print(f"  Spanish passage: {es} tokens")
    print(
        f"  A {CTX:,}-token window holds about "
        f"{CTX // en:,} English copies or {CTX // es:,} Spanish copies."
    )
    print(
        f"  Spanish input cost multiplier: {multiplier:.2f}x "
        "(assuming the same per-token input price)."
    )

print(
    "\nProduct impact: A multilingual product should calculate token budgets "
    "with the tokenizer used by the target model instead of assuming that "
    "equivalent messages consume equal context or cost across languages."
)



cl100k_base
  English passage: 143 tokens
  Spanish passage: 197 tokens
  A 128,000-token window holds about 895 English copies or 649 Spanish copies.
  Spanish input cost multiplier: 1.38x (assuming the same per-token input price).

o200k_base
  English passage: 143 tokens
  Spanish passage: 160 tokens
  A 128,000-token window holds about 895 English copies or 800 Spanish copies.
  Spanish input cost multiplier: 1.12x (assuming the same per-token input price).

Product impact: A multilingual product should calculate token budgets with the tokenizer used by the target model instead of assuming that equivalent messages consume equal context or cost across languages.


## Part 4: Bias splits and one failure

### A. Three Spanish/English word splits

To avoid selecting examples by guesswork, the cell below evaluates a small set of faithful English/Spanish word pairs with `cl100k_base`, ranks them by the additional number of tokens required by the Spanish form, and displays the three strongest examples. This makes the selection reproducible from the tokenizer itself.

The point is not that Spanish words are inherently inefficient. BPE-style tokenizers learn reusable pieces from their training data, so vocabulary frequency, morphology, accents, and learned merge patterns can cause equivalent words to fragment differently.

### B. Counterintuitive failure case

The Spanish word `información` provides a useful Unicode normalization case. Two versions can look identical to a reader while using different underlying Unicode code points: one can store `ó` as a single character, while another can store `o` followed by a combining accent. The code below compares those representations with both tokenizers to see whether visually identical text receives the same tokenization.


In [5]:
candidate_pairs = [
    ("storage", "almacenamiento"),
    ("tools", "herramientas"),
    ("developers", "desarrolladores"),
    ("learning", "aprendizaje"),
    ("responsibility", "responsabilidad"),
    ("availability", "disponibilidad"),
    ("understanding", "comprensión"),
    ("efficiency", "eficiencia"),
    ("implementation", "implementación"),
    ("security", "seguridad"),
    ("requirements", "requisitos"),
    ("environment", "entorno"),
]

ranked_pairs = sorted(
    candidate_pairs,
    key=lambda pair: (
        count_tokens(pair[1], gpt4) - count_tokens(pair[0], gpt4),
        count_tokens(pair[1], gpt4) / count_tokens(pair[0], gpt4),
    ),
    reverse=True,
)

bias_pairs = [
    pair
    for pair in ranked_pairs
    if count_tokens(pair[1], gpt4) > count_tokens(pair[0], gpt4)
][:3]

assert len(bias_pairs) == 3, "Need three Spanish words that fragment more than English."

print("Three strongest cl100k_base examples:")
for english_word, spanish_word in bias_pairs:
    print(f"\nEnglish: {english_word}")
    show_split(english_word, gpt4)
    print(f"Spanish: {spanish_word}")
    show_split(spanish_word, gpt4)

print(
    "\nExplanation: In each selected pair, the Spanish form requires more "
    "cl100k_base tokens than its English equivalent. The tokenizer has learned "
    "different subword merges for the two languages, so equivalent concepts "
    "do not necessarily have equal token cost."
)

# Failure case: visually identical text with different Unicode representations.
import unicodedata

failure_text = "información"
decomposed_text = unicodedata.normalize("NFD", failure_text)

print("\nFailure case: visually identical Unicode text")
print("NFC display:", failure_text)
print("NFD display:", decomposed_text)
print("Same Python string:", failure_text == decomposed_text)

print("\nUnderlying Unicode code points:")
print("NFC:", [f"U+{ord(ch):04X}" for ch in failure_text])
print("NFD:", [f"U+{ord(ch):04X}" for ch in decomposed_text])

for encoder_name, encoder in [
    ("cl100k_base", gpt4),
    ("o200k_base", gpt4o),
]:
    print(f"\n{encoder_name}")

    for label, text in [
        ("NFC", failure_text),
        ("NFD", decomposed_text),
    ]:
        token_ids = encoder.encode(text)
        byte_pieces = [
            encoder.decode_single_token_bytes(token_id)
            for token_id in token_ids
        ]

        print(f"  {label}: {len(token_ids)} token(s)")
        print(f"    token IDs: {token_ids}")
        print(f"    raw byte pieces: {byte_pieces}")

print(
    "\nWhy this is surprising: The two versions of información look the same "
    "to a reader, but they are stored as different Unicode code-point sequences. "
    "Because tokenizers operate on the underlying encoded text rather than its "
    "visual appearance, the representations can produce different tokenization."
)

print(
    "\nMitigation: Normalize user text to a consistent Unicode form, such as NFC, "
    "when doing so preserves the intended meaning. Token budgets should still be "
    "calculated with the tokenizer used by the target model rather than estimated "
    "from visible character or word counts."
)


Three strongest cl100k_base examples:

English: tools
'tools'                  -> 1 token(s): ['tools']
Spanish: herramientas
'herramientas'           -> 4 token(s): ['h', 'err', 'amient', 'as']

English: developers
'developers'             -> 1 token(s): ['developers']
Spanish: desarrolladores
'desarrolladores'        -> 4 token(s): ['des', 'ar', 'roll', 'adores']

English: learning
'learning'               -> 1 token(s): ['learning']
Spanish: aprendizaje
'aprendizaje'            -> 4 token(s): ['ap', 'rend', 'iz', 'aje']

Explanation: In each selected pair, the Spanish form requires more cl100k_base tokens than its English equivalent. The tokenizer has learned different subword merges for the two languages, so equivalent concepts do not necessarily have equal token cost.

Failure case: visually identical Unicode text
NFC display: información
NFD display: información
Same Python string: False

Underlying Unicode code points:
NFC: ['U+0069', 'U+006E', 'U+0066', 'U+006F', 'U+0072', 'U+

## Part 5: Submission checklist

Before submission:

- Run the notebook from top to bottom and confirm every cell succeeds.
- Keep the resulting outputs so the measured token counts and splits are visible.
- Create one repository issue logging the experiment as a research note.
- Update the pull-request description with one paragraph summarizing the measured headline numbers.
- Confirm the notebook, issue, and pull request all refer to the same Spanish/English experiment.

The assignment rubric evaluates repository quality, counts from both tokenizers, multilingual-tax calculation, three bias splits, cost/context figures, the failure case, and pull-request hygiene.
